# CS 839 HW2 — When does a team beat one agent?

Three systems answer the same 8 data-analysis tasks over the multi-file flights dataset in `data/`
(generated by `generate_data.py`; hidden answer key in `tasks.json`). All use the **same model** and the
**same budget: at most 20 model calls per task** (every LLM call counts, including tool-loop steps).
A pilot with 11 calls (3 per team agent) is kept in `results_pilot.csv` / `traces_pilot/`: team agents ran out of
calls before computing anything, so the budget was raised and unused calls now roll over.

**Two conditions.** Run 1 used the model's default reasoning effort; every system scored ~100%, so run 2 sets
`REASONING_EFFORT = "low"` for **every** agent (same model, less deliberation per call). Rows and traces are tagged
with `effort`, and the results tables compare `system @ effort`.

| system | topology | calls |
|---|---|---|
| `single` | one tool-using agent in one conversation; it may check/revise its own work before submitting | ≤ 20 |
| `hierarchy` (A) | manager plans (no code) → ≤3 workers (own files only) → **same manager** synthesizes | 1 + 18 (workers) + 1 |
| `debate` (B) | 3 independent solvers (all files) → critic (sees code, no data) → aggregator | 18 (solvers) + 1 + 1 |

Worker / solver calls come from a pool of 18 with **rollover**: each agent gets an equal base share
(18 / number of agents; 6 when there are 3), and calls it leaves unused are added to the next agent's cap. Each agent
is told its own cap. (Agents run sequentially, so this is possible; a truly parallel deployment would have to fix
per-agent caps up front.)

**Workers, solvers, critic and aggregator are one-shot**: a fresh context (and, if they run code, a private
workspace with only the files they may see), discarded after they submit. The **manager keeps one conversation
for the whole task** (plan → read reports → final answer). Every agent is created fresh **per task**, so nothing
carries over between tasks. Agents never talk directly — Python passes their JSON outputs along (the same
pattern as the in-class `assignment4.ipynb`).

In [1]:
import os, io, re, json, time, getpass, shutil, tempfile, contextlib, traceback
from dataclasses import dataclass
from pathlib import Path
import pandas as pd
from openai import OpenAI

### Configuration

In [5]:
MODEL = os.getenv("CS839_MODEL", "gpt-5.6-luna")
if not os.getenv("OPENAI_API_KEY"):
    key = getpass.getpass("Paste your team's API key (hidden): ").strip()
    if key:
        os.environ["OPENAI_API_KEY"] = key

HW2 = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd() / "HW2"
DATA_DIR, TASKS_PATH, TRACE_DIR = HW2 / "data", HW2 / "tasks.json", HW2 / "traces"
TRACE_DIR.mkdir(exist_ok=True)

# ---- budget (identical across systems) ----
BUDGET_CALLS      = 20      # max model calls per task, every system (pilot used 11 -> team agents starved)
MAX_OUTPUT_TOKENS = 4000    # per call (reasoning tokens count toward this)
REASONING_EFFORT  = "low"   # applied to EVERY agent. None = model default (run 1); "low" = run 2
# Team agents draw from a shared pool with ROLLOVER: each gets an equal base share, and calls an agent
# does not use carry over to the next agent (agents run one after another, so nothing is stranded).
MAX_WORKERS = 3
WORKER_POOL = BUDGET_CALLS - 2       # hierarchy: manager plan (1) + workers (18, split over 1-3 workers) + manager final (1)
N_SOLVERS   = 3
SOLVER_POOL = BUDGET_CALLS - 2       # debate: solvers (18 -> 6 base each) + critic (1) + aggregator (1)

# ---- pricing: $ per 1M tokens. FILL IN for MODEL (cost columns stay NaN until set) ----
PRICE_IN_PER_M  = 0.1
PRICE_OUT_PER_M = 0.5

README_TEXT = (DATA_DIR / "README.md").read_text()
WEATHER_CODES_TEXT = (DATA_DIR / "weather_codes.md").read_text()   # agents must open the file; only the critic gets it inline
ALL_FILES = sorted(str(p.relative_to(DATA_DIR)) for p in DATA_DIR.rglob("*")
                   if p.is_file() and not any(part.startswith(".") for part in p.relative_to(DATA_DIR).parts))

# tasks.json holds the expected answers -> used ONLY by the grader, never shown to any agent
TASKS = json.loads(TASKS_PATH.read_text())["tasks"]
TASK_BY_ID = {t["id"]: t for t in TASKS}

ANSWER_FORMATS = {
    "T1": '{"WN": 12.3, "DL": 12.3, "AA": 12.3, "UA": 12.3, "B6": 12.3, "AS": 12.3}   (delay rate in %, 1 decimal)',
    "T2": '{"WN": {"busiest": "ATL-JFK", "worst": "ATL-JFK"}, "DL": {...}, "AA": {...}, "UA": {...}, "B6": {...}, "AS": {...}}   '
          '(a route = the two airport codes in alphabetical order joined by "-")',
    "T3": '{"pct_in_bad_weather": 12.3}   (%, 1 decimal)',
    "T4": '{"bad_weather_rate": 12.3, "other_rate": 12.3}   (delay rates in %, 1 decimal)',
    "T5": '{"network_rate": 12.3}   (delay rate in %, 1 decimal)',
    "T6": '{"top5": ["ATL-JFK", "...", "...", "...", "..."]}   (most flights first; a route = the two airport codes '
          'in alphabetical order joined by "-")',
    "T7": '{"cancelled": 0}',
    "T8": '{"avg_delay_min": 12.3}   (minutes, 1 decimal)',
}

def task_message(task):
    return (f"QUESTION ({task['id']}):\n{task['prompt']}\n\n"
            f"REQUIRED ANSWER FORMAT (JSON):\n{ANSWER_FORMATS[task['id']]}")

print(f"model={MODEL}  tasks={len(TASKS)}  files={len(ALL_FILES)}  budget={BUDGET_CALLS} calls/task")

model=gpt-5.6-luna  tasks=8  files=10  budget=20 calls/task


### LLM client + run ledger
`RunLog` is the ledger for one (system, task, repeat) run: every model call (agent, tokens, seconds), every
inter-agent handoff (who → whom, characters), and a full event trace. It also **enforces the call budget**:
`LLMClient.respond` raises `BudgetExceeded` once the run has used `BUDGET_CALLS` calls.

In [6]:
class BudgetExceeded(Exception):
    pass

@dataclass
class ToolCall:
    call_id: str
    name: str
    arguments: dict

@dataclass
class LLMResponse:
    text: str
    tool_calls: list
    output: list          # raw output items, appended to the agent's own input on the next turn
    input_tokens: int
    output_tokens: int
    status: str


def cost_usd(input_tokens, output_tokens):
    if PRICE_IN_PER_M is None or PRICE_OUT_PER_M is None:
        return float("nan")
    return input_tokens / 1e6 * PRICE_IN_PER_M + output_tokens / 1e6 * PRICE_OUT_PER_M


class RunLog:
    def __init__(self, system, task_id, repeat=0, max_calls=BUDGET_CALLS):
        self.system, self.task_id, self.repeat, self.max_calls = system, task_id, repeat, max_calls
        self.calls, self.messages, self.events = [], [], []

    @property
    def n_calls(self):
        return len(self.calls)

    @property
    def remaining(self):
        return self.max_calls - self.n_calls

    def charge(self, agent, resp, seconds):
        self.calls.append(dict(agent=agent, input_tokens=resp.input_tokens, output_tokens=resp.output_tokens,
                               seconds=round(seconds, 2), status=resp.status))

    def handoff(self, frm, to, content):
        self.messages.append(dict(frm=frm, to=to, chars=len(content)))
        self.event(frm, "handoff", to=to, content=content)

    def event(self, agent, kind, **data):
        self.events.append(dict(agent=agent, kind=kind, **data))

    def totals(self):
        i = sum(c["input_tokens"] for c in self.calls)
        o = sum(c["output_tokens"] for c in self.calls)
        return dict(calls=self.n_calls, input_tokens=i, output_tokens=o, cost_usd=cost_usd(i, o),
                    seconds=round(sum(c["seconds"] for c in self.calls), 1),
                    handoffs=len(self.messages), comm_chars=sum(m["chars"] for m in self.messages))


class LLMClient:
    def __init__(self, model=None):
        self.client = OpenAI()
        self.model = model or MODEL

    def respond(self, instructions, input, *, log, agent, tools=None, tool_choice="auto", json_mode=False):
        if log.remaining <= 0:
            raise BudgetExceeded(f"{log.system}/{log.task_id}: {log.max_calls} calls used")
        if isinstance(input, str):
            input = [{"role": "user", "content": input}]
        kwargs = dict(model=self.model, instructions=instructions, input=input, max_output_tokens=MAX_OUTPUT_TOKENS)
        if tools:
            kwargs.update(tools=tools, tool_choice=tool_choice)
        if json_mode:
            kwargs["text"] = {"format": {"type": "json_object"}}
        if REASONING_EFFORT:
            kwargs["reasoning"] = {"effort": REASONING_EFFORT}
        t0 = time.perf_counter()
        resp = self.client.responses.create(**kwargs)
        seconds = time.perf_counter() - t0

        items = resp.output or []
        tool_calls = []
        for it in items:
            if getattr(it, "type", None) == "function_call":
                try:
                    args = json.loads(it.arguments)
                except (json.JSONDecodeError, TypeError):
                    args = {}
                tool_calls.append(ToolCall(it.call_id, it.name, args))
        u = resp.usage
        out = LLMResponse(text=resp.output_text or "", tool_calls=tool_calls, output=items,
                          input_tokens=u.input_tokens if u else 0, output_tokens=u.output_tokens if u else 0,
                          status=getattr(resp, "status", None) or "completed")
        log.charge(agent, out, seconds)
        return out


def parse_json(text):
    # Best-effort JSON object extraction from model text.
    if isinstance(text, dict):
        return text
    if not text:
        return None
    try:
        obj = json.loads(text)
        return obj if isinstance(obj, dict) else None
    except (json.JSONDecodeError, TypeError):
        m = re.search(r"\{.*\}", text, re.S)
        if m:
            try:
                obj = json.loads(m.group(0))
                return obj if isinstance(obj, dict) else None
            except json.JSONDecodeError:
                return None
    return None

### Tools and information boundaries
Each agent gets a `Workspace`: a private temp directory containing **only the files it is allowed to see**.
`run_python` executes with that directory as the working directory, so `pd.read_csv("airlines/DL.csv")` works
for an assigned file and raises `FileNotFoundError` for anything else. Variables persist across one agent's
`run_python` calls (not across agents). `submit_answer` ends the agent's turn.

In [7]:
class Workspace:
    def __init__(self, files, max_output_chars=4000):
        self.files = sorted(files)
        self.root = Path(tempfile.mkdtemp(prefix="hw2_ws_"))
        for f in self.files:
            dst = self.root / f
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy(DATA_DIR / f, dst)
        self.ns = {"pd": pd}
        self.max_output_chars = max_output_chars

    def run_python(self, code):
        buf = io.StringIO()
        try:
            with contextlib.chdir(self.root), contextlib.redirect_stdout(buf):
                exec(code, self.ns)
        except Exception:
            return {"ok": False, "stdout": buf.getvalue()[-1500:], "error": traceback.format_exc(limit=2)[-1500:]}
        out = buf.getvalue()
        if len(out) > self.max_output_chars:
            out = out[: self.max_output_chars] + "\n...[truncated]"
        return {"ok": True, "stdout": out}

    def close(self):
        shutil.rmtree(self.root, ignore_errors=True)


TOOL_SCHEMAS = [
    {"type": "function", "name": "run_python",
     "description": ("Execute Python in your workspace. The working directory contains only your files "
                     "(use relative paths, e.g. 'airlines/DL.csv'). pandas is available as pd. Variables persist "
                     "between your calls. Use print() to see results."),
     "parameters": {"type": "object", "properties": {"code": {"type": "string"}}, "required": ["code"]}},
    {"type": "function", "name": "submit_answer",
     "description": "Submit your final result. This ends your turn.",
     "parameters": {"type": "object", "properties": {
         "answer_json": {"type": "string", "description": "Your result as a JSON object, in the requested format."},
         "notes": {"type": "string", "description": "Briefly: method, assumptions, and any data issues you handled."}},
         "required": ["answer_json", "notes"]}},
]
FORCE_SUBMIT = {"type": "function", "name": "submit_answer"}

### Tool-using agent (stateless)
One `ToolAgent` = one fresh conversation + one private workspace + its own call cap. On its **last allowed call**
`submit_answer` is forced, so an agent always returns something. Every `run_python` result also reports
`calls_remaining`, so an agent can plan to compute and submit before it is cut off.

In [8]:
class ToolAgent:
    def __init__(self, llm, name, instructions, files, max_calls, log):
        self.llm, self.name, self.instructions, self.max_calls, self.log = llm, name, instructions, max_calls, log
        self.ws = Workspace(files)
        self.input, self.calls_used, self.code_log = [], 0, []
        self.result = None       # {"answer": dict|None, "raw": str, "notes": str}

    def run(self, user_msg):
        self.input.append({"role": "user", "content": user_msg})
        return self._loop()

    def _loop(self):
        while self.calls_used < self.max_calls:
            last = self.calls_used == self.max_calls - 1 or self.log.remaining == 1
            try:
                resp = self.llm.respond(self.instructions, self.input, log=self.log, agent=self.name,
                                        tools=TOOL_SCHEMAS, tool_choice=FORCE_SUBMIT if last else "auto")
            except BudgetExceeded:
                self.log.event(self.name, "budget_exceeded")
                break
            self.calls_used += 1
            self.input += resp.output
            if not resp.tool_calls:
                self.log.event(self.name, "text", text=resp.text[:2000])
                self.input.append({"role": "user", "content": "Use run_python to compute, then call submit_answer."})
                continue
            submitted = False
            for tc in resp.tool_calls:
                if tc.name == "run_python":
                    code = tc.arguments.get("code", "")
                    out = self.ws.run_python(code)
                    self.code_log.append(code)
                    self.log.event(self.name, "run_python", code=code, result=out)
                    payload = {**out, "calls_remaining": min(self.max_calls - self.calls_used, self.log.remaining)}
                elif tc.name == "submit_answer":
                    raw = tc.arguments.get("answer_json", "")
                    self.result = {"answer": parse_json(raw), "raw": raw, "notes": tc.arguments.get("notes", "")}
                    self.log.event(self.name, "submit", **self.result)
                    payload, submitted = {"received": True}, True
                else:
                    payload = {"ok": False, "error": f"unknown tool {tc.name}"}
                self.input.append({"type": "function_call_output", "call_id": tc.call_id, "output": json.dumps(payload)})
            if submitted:
                break
        return self.result

    def close(self):
        self.ws.close()

### Prompts
The data README (conventions + caveats) goes to every agent that plans or solves. `weather_codes.md` is a file
in the workspace: an agent has to open it (or be told its rule by the manager).

In [9]:
SOLVER_PROMPT = """
You are a careful data analyst answering a question about airline operations data.
Compute every number with the run_python tool -- never guess or use prior knowledge for numbers.

DATA DOCUMENTATION
{readme}

Your workspace contains: {files}
You have at most {max_calls} model calls including the final submit_answer (each tool result shows
calls_remaining; your last call can only be submit_answer), so combine several steps in each run_python call. Inspect the data before trusting it, and sanity-check your result against the documentation and
the question before you submit. Then call submit_answer with answer_json in exactly the requested format.
""".strip()

MANAGER_PROMPT = """
You are the MANAGER of a small data-analysis team. You do not run code. You work in two phases.

PHASE 1 -- PLAN. Split the question into at most {max_workers} subtasks for WORKER agents, who run code
independently and in parallel.

What a worker knows: ONLY the instructions you write and the files you assign. Workers do not see the question,
the data documentation, or each other. Give each worker every definition and data caveat it needs.
Workers share a pool of {worker_pool} model calls, split evenly across the workers you create; calls a worker
does not use roll over to the next worker. Keep each subtask focused.
In phase 2 YOU will combine the workers' reports into the final answer, so ask workers to return what you will
need. Use as few workers as the question warrants (1 is fine for a simple question).
Return a JSON object:
{{"subtasks": [{{"worker": "w1", "files": ["<path from the file list>", ...], "instructions": "...",
                 "return_format": "<JSON shape the worker must submit>"}}],
  "synthesis_plan": "<how you will combine the reports>"}}

PHASE 2 -- SYNTHESIZE. You will then receive the workers' reports. Combine them into the final answer to the
question. Treat the reports as fallible: check them against your plan, the data documentation and each other.
Return a JSON object: {{"answer": <the answer in the required format>, "rationale": "<short>"}}
""".strip()

MANAGER_SYNTH_MSG = "PHASE 2. Your workers' reports (fallible) are below. Produce the final answer as JSON.\n\n"

WORKER_PROMPT = """
You are a WORKER on a data-analysis team. The manager gave you one subtask. Compute it with the run_python tool
-- never guess numbers. Your workspace contains only: {files}
You have at most {max_calls} model calls including the final submit_answer (each tool result shows
calls_remaining; your last call can only be submit_answer), so combine steps. Submit your
result with submit_answer in the requested return format; use notes for your method and any data issues you saw.
""".strip()

# Not used by the main hierarchy (the manager synthesizes). Kept for a possible topology surgery:
# hand synthesis to a fresh agent that never saw the plan.
SYNTH_PROMPT = """
You are the SYNTHESIZER of a data-analysis team. Workers computed parts of the answer from the data; you cannot
run code. Combine their reports into the final answer to the question. Treat the reports as fallible: check them
against the data documentation and against each other.
Return a JSON object: {"answer": <the answer in the required format>, "rationale": "<short>"}
""".strip()

CRITIC_PROMPT = """
You are a CRITIC. Three analysts answered the same question independently. You cannot run code, but you see each
analyst's code, result and notes, plus the data documentation. Find concrete errors: misapplied definitions,
data issues not handled, computation bugs, wrong format. Say where the analysts disagree and whose method is
best supported.
Return a JSON object: {"issues": [{"analyst": "S1", "problem": "..."}], "disagreements": "...",
                       "best": "S1|S2|S3|none", "correction": "<what the final answer should change, if anything>"}
""".strip()

AGG_PROMPT = """
You are the AGGREGATOR. Three analysts answered the same question independently and a critic reviewed them.
Produce the single final answer. Treat every input as fallible; prefer the answer whose method is sound over
the majority. You cannot run code, so any value you output must come from an analyst's reported results.
Return a JSON object: {"answer": <the answer in the required format>, "chosen": "S1|S2|S3|combined",
                       "rationale": "<short>"}
""".strip()


def file_catalog(files=ALL_FILES):
    # What the manager sees about the data without running code: CSV headers + row counts, doc titles.
    lines = []
    for f in files:
        p = DATA_DIR / f
        if p.suffix == ".csv":
            with open(p) as fh:
                header = fh.readline().strip()
                n = sum(1 for _ in fh)
            lines.append(f"- {f}  ({n} rows)  columns: {header}")
        else:
            lines.append(f"- {f}  (document: {p.read_text().splitlines()[0].lstrip('# ')})")
    return "\n".join(lines)

## The three systems
Each returns `(final_answer_dict_or_None, metrics_dict)` and records everything in the `RunLog`.

In [10]:
from collections import Counter

def unwrap_answer(obj):
    # Accept {"answer": {...}} or the bare answer dict.
    if isinstance(obj, dict) and isinstance(obj.get("answer"), dict):
        return obj["answer"]
    return obj if isinstance(obj, dict) else None


# ---------------------------------------------------------------- single agent
def run_single(task, llm, log):
    # One agent, one conversation, the whole budget. It decides itself when to verify and when to submit.
    agent = ToolAgent(llm, "single", SOLVER_PROMPT.format(readme=README_TEXT, files=", ".join(ALL_FILES),
                                                          max_calls=BUDGET_CALLS),
                      ALL_FILES, BUDGET_CALLS, log)
    try:
        res = agent.run(task_message(task))
        n_code = len(agent.code_log)
    finally:
        agent.close()
    return (res["answer"] if res else None), {"code_runs": n_code}


# ---------------------------------------------------------------- A: hierarchy
CAVEAT_PATTERNS = {   # which README / weather caveats did the manager pass down to workers?
    "threshold_15": r"\b15\b", "cancelled": r"cancel", "utc": r"utc|time ?zone|offset",
    "dedup": r"duplicat|unique|repeated", "midnight": r"midnight|next day|day after",
    "bad_weather_rule": r"\+RA|visibility|weather_codes",
}

AIRLINE_FILES = [f for f in ALL_FILES if f.startswith("airlines/")]
FEW_WORKERS_RULE = "Use as few workers as the question warrants (1 is fine for a simple question)."
# Surgery #4: the manager may no longer choose 1 worker -- it must partition the airline files across workers.
SPLIT_RULE = (f"You MUST use exactly {MAX_WORKERS} workers and split the airline files between them: every file in "
              f"airlines/ goes to exactly one worker ({len(AIRLINE_FILES) // MAX_WORKERS} carriers each). Shared files "
              "(weather.csv, airports.csv, the docs) may be given to any worker that needs them.")

def run_hierarchy(task, llm, log, split=False):
    m = {"plan_ok": False, "n_workers": 0, "invalid_files": [], "files_overlap": 0, "workers_no_result": 0,
         "worker_caps": [], "worker_calls_used": [],
         "caveats_passed": []}
    # 1) manager, phase 1: plan. The manager's conversation (mgr_hist) is kept for phase 2.
    mgr_prompt = MANAGER_PROMPT.format(max_workers=MAX_WORKERS, worker_pool=WORKER_POOL)
    if split:
        assert FEW_WORKERS_RULE in mgr_prompt
        mgr_prompt = mgr_prompt.replace(FEW_WORKERS_RULE, SPLIT_RULE)
    mgr_hist = [{"role": "user", "content": task_message(task) + "\n\nDATA DOCUMENTATION\n" + README_TEXT
                 + "\n\nFILES AVAILABLE TO ASSIGN\n" + file_catalog() + "\n\nPHASE 1: return your plan as JSON."}]
    resp = llm.respond(mgr_prompt, mgr_hist, log=log, agent="manager", json_mode=True)
    mgr_hist += resp.output
    plan = parse_json(resp.text) or {}
    log.event("manager", "plan", plan=plan, raw=resp.text[:4000])
    subtasks = []
    for st in (plan.get("subtasks") or [])[:MAX_WORKERS]:
        files = [f.removeprefix("data/") for f in st.get("files", []) if isinstance(f, str)]
        m["invalid_files"] += [f for f in files if f not in ALL_FILES]
        files = [f for f in files if f in ALL_FILES]
        if files and st.get("instructions"):
            subtasks.append({**st, "files": files})
    if not subtasks:
        return None, m
    m.update(plan_ok=True, n_workers=len(subtasks))
    assigned = [f for st in subtasks for f in st["files"]]
    m["files_overlap"] = len(assigned) - len(set(assigned))
    all_instr = " ".join(st["instructions"] + " " + str(st.get("return_format", "")) for st in subtasks)
    m["caveats_passed"] = [k for k, pat in CAVEAT_PATTERNS.items() if re.search(pat, all_instr, re.I)]
    m["airlines_per_worker"] = [sum(f in AIRLINE_FILES for f in st["files"]) for st in subtasks]
    m["split_ok"] = (len(subtasks) == MAX_WORKERS and sorted(f for f in assigned if f in AIRLINE_FILES) == sorted(AIRLINE_FILES))

    # 2) workers: fresh agents, only their files + the manager's instructions
    reports, carry = [], 0
    base = WORKER_POOL // len(subtasks)
    for i, st in enumerate(subtasks, 1):
        wname = st.get("worker") or f"w{i}"
        cap = base + carry                                   # rollover: unused calls of earlier workers
        msg = f"SUBTASK\n{st['instructions']}\n\nRETURN FORMAT (JSON)\n{st.get('return_format', 'a JSON object')}"
        log.handoff("manager", wname, msg)
        w = ToolAgent(llm, wname, WORKER_PROMPT.format(files=", ".join(st["files"]), max_calls=cap),
                      st["files"], cap, log)
        try:
            res = w.run(msg)
        finally:
            w.close()
        carry = cap - w.calls_used
        m["worker_caps"].append(cap)
        m["worker_calls_used"].append(w.calls_used)
        if not res:
            m["workers_no_result"] += 1
        reports.append({"worker": wname, "files": st["files"], "instructions": st["instructions"],
                        "result": res["raw"] if res else "NO RESULT (worker did not submit)",
                        "notes": res["notes"] if res else ""})

    # 3) manager, phase 2: same conversation (it still sees its own plan) + the reports
    rep_text = json.dumps(reports, indent=1)
    log.handoff("workers", "manager", rep_text)
    mgr_hist.append({"role": "user", "content": MANAGER_SYNTH_MSG + rep_text})
    try:
        resp = llm.respond(mgr_prompt, mgr_hist, log=log, agent="manager", json_mode=True)
    except BudgetExceeded:
        return None, m
    out = parse_json(resp.text)
    log.event("manager", "final", output=out, raw=resp.text[:4000])
    return unwrap_answer(out), m


# ---------------------------------------------------------------- B: solvers -> critic -> aggregator
def solve_independently(task, llm, log):
    # N_SOLVERS fresh solvers, same prompt and files, shared pool with rollover. Used by debate and debate_vote.
    proposals, carry, caps, used = [], 0, [], []
    base = SOLVER_POOL // N_SOLVERS
    for i in range(1, N_SOLVERS + 1):
        name = f"S{i}"
        cap = base + carry                                   # rollover: unused calls of earlier solvers
        s = ToolAgent(llm, name, SOLVER_PROMPT.format(readme=README_TEXT, files=", ".join(ALL_FILES), max_calls=cap),
                      ALL_FILES, cap, log)
        try:
            res = s.run(task_message(task))
            code_txt = "\n# ----\n".join(s.code_log)
        finally:
            s.close()
        carry = cap - s.calls_used
        caps.append(cap)
        used.append(s.calls_used)
        proposals.append({"analyst": name, "answer": res["answer"] if res else None,
                          "raw": res["raw"] if res else "NO RESULT", "notes": res["notes"] if res else "",
                          "code": code_txt[-3000:]})

    canons = [canon(task["id"], p["answer"]) for p in proposals]
    valid = [c for c in canons if c is not None]
    majority = max(set(valid), key=valid.count) if valid else None
    m = {"distinct_answers": len(set(valid)), "solvers_no_result": canons.count(None),
         "solver_caps": caps, "solver_calls_used": used,
         "solver_scores": [grade(task["id"], p["answer"], task["expected"])[0] for p in proposals]}  # analysis only
    return proposals, canons, majority, m


def run_debate(task, llm, log):
    proposals, canons, majority, m = solve_independently(task, llm, log)

    # critic: sees code + results + docs, no data
    crit_view = [{k: p[k] for k in ("analyst", "raw", "notes", "code")} for p in proposals]
    crit_text = json.dumps(crit_view, indent=1)
    log.handoff("solvers", "critic", crit_text)
    crit_in = (task_message(task) + "\n\nDATA DOCUMENTATION\n" + README_TEXT + "\n\nweather_codes.md\n"
               + WEATHER_CODES_TEXT + "\n\nANALYST SUBMISSIONS\n" + crit_text)
    resp = llm.respond(CRITIC_PROMPT, crit_in, log=log, agent="critic", json_mode=True)
    critique = parse_json(resp.text) or {"raw": resp.text[:3000]}
    log.event("critic", "critique", critique=critique)

    # aggregator: answers + notes + critique
    agg_view = json.dumps([{k: p[k] for k in ("analyst", "raw", "notes")} for p in proposals], indent=1)
    crit_json = json.dumps(critique, indent=1)
    log.handoff("solvers", "aggregator", agg_view)
    log.handoff("critic", "aggregator", crit_json)
    agg_in = (task_message(task) + "\n\nANALYST ANSWERS\n" + agg_view + "\n\nCRITIQUE\n" + crit_json)
    try:
        resp = llm.respond(AGG_PROMPT, agg_in, log=log, agent="aggregator", json_mode=True)
    except BudgetExceeded:
        return None, m
    out = parse_json(resp.text)
    log.event("aggregator", "final", output=out, raw=resp.text[:4000])
    final = unwrap_answer(out)
    fc = canon(task["id"], final)
    m.update(critic_best=critique.get("best"), agg_chosen=(out or {}).get("chosen"),
             final_is_majority=fc is not None and fc == majority,
             final_matches_a_solver=fc in canons if fc is not None else False)
    return final, m


# ---------------------------------------------------------------- surgery #1: debate without critic/aggregator
def run_debate_vote(task, llm, log):
    # Same 3 solvers; the critic and aggregator nodes are removed. Final = plurality of the solvers' answers
    # (compared after canonicalisation); if all differ, the first valid solver (S1) wins. No LLM review at all.
    proposals, canons, majority, m = solve_independently(task, llm, log)
    valid = [(c, p) for c, p in zip(canons, proposals) if c is not None]
    if not valid:
        return None, m
    counts = Counter(c for c, _ in valid)              # insertion-ordered -> ties go to the earliest solver
    win, n = counts.most_common(1)[0]
    final = next(p["answer"] for c, p in valid if c == win)
    m.update(vote_count=n, vote_tie=(n == 1 and len(valid) > 1), final_is_majority=True,
             final_matches_a_solver=True)
    log.event("vote", "final", winner=next(p["analyst"] for c, p in valid if c == win), votes=n, output=final)
    return final, m


# ---------------------------------------------------------------- surgery #4: hierarchy with a forced split
def run_hierarchy_split(task, llm, log):
    return run_hierarchy(task, llm, log, split=True)


SYSTEMS = {"single": run_single, "hierarchy": run_hierarchy, "debate": run_debate,
           "debate_vote": run_debate_vote, "hierarchy_split": run_hierarchy_split}

## Grader (programmatic, no LLM judge)
Numbers must be within ±0.1 of the reference (counts exact). Partial credit = fraction of fields correct
(T1: 6 carriers, T2: 12 route picks, T4: 2 rates, T6: overlap with the true top-5 set).
A wrong answer is also matched against the generator's `trap_answers` to label *which* mistake was made.

In [11]:
CARRIER_ORDER = ["WN", "DL", "AA", "UA", "B6", "AS"]

def norm_route(r):
    parts = re.split(r"\s*(?:->|→|–|-|/|\bTO\b)\s*", str(r).strip().upper())
    return "-".join(sorted(p for p in parts if p))

def num_ok(a, e, tol=0.1):
    try:
        return abs(float(a) - float(e)) <= tol + 1e-9
    except (TypeError, ValueError):
        return False

def _get(d, k):
    if not isinstance(d, dict):
        return None
    for kk in (k, k.lower(), k.upper()):
        if kk in d:
            return d[kk]
    return None

def grade(tid, ans, exp):
    # Return (score in [0,1], per-field detail). `exp` may be the true answer or a trap answer.
    ans = unwrap_answer(ans) if isinstance(ans, dict) else None
    if ans is None:
        return 0.0, {"error": "no answer"}
    checks = {}
    if tid == "T1":
        checks = {cc: num_ok(_get(ans, cc), exp[cc]) for cc in CARRIER_ORDER if cc in exp}
    elif tid == "T2":
        for cc in CARRIER_ORDER:
            a = _get(ans, cc) or {}
            for f in ("busiest", "worst"):
                checks[f"{cc}.{f}"] = bool(a.get(f)) and norm_route(a.get(f)) == norm_route(exp[cc][f])
    elif tid == "T3":
        checks = {"pct_in_bad_weather": num_ok(_get(ans, "pct_in_bad_weather"), exp["pct_in_bad_weather"])}
    elif tid == "T4":
        checks = {k: num_ok(_get(ans, k), exp[k]) for k in ("bad_weather_rate", "other_rate") if k in exp}
    elif tid == "T5":
        checks = {"network_rate": num_ok(_get(ans, "network_rate"), exp["network_rate"])}
    elif tid == "T6":
        got = {norm_route(r) for r in (_get(ans, "top5") or [])[:5]}
        want = {norm_route(r) for r in exp["top5"]}
        return len(got & want) / 5, {"missing": sorted(want - got), "extra": sorted(got - want)}
    elif tid == "T7":
        checks = {"cancelled": num_ok(_get(ans, "cancelled"), exp["cancelled"], tol=0)}
    elif tid == "T8":
        checks = {"avg_delay_min": num_ok(_get(ans, "avg_delay_min"), exp["avg_delay_min"])}
    return (sum(checks.values()) / len(checks) if checks else 0.0), checks

def match_trap(task, ans, score):
    # Name the trap whose answer the agent reproduced (exact if 'trap', better-than-truth if 'partial:trap').
    if score >= 1.0 or ans is None:
        return ""
    best, best_s = "", score
    for name, trap_ans in task.get("trap_answers", {}).items():
        s, _ = grade(task["id"], ans, {**task["expected"], **trap_ans})
        if s > best_s:
            best, best_s = name, s
    return "" if not best else (best if best_s >= 1.0 else f"partial:{best}")

def canon(tid, ans):
    # Canonical string of an answer (rounded numbers, normalized routes) -- to measure solver agreement.
    ans = unwrap_answer(ans) if isinstance(ans, dict) else None
    if ans is None:
        return None
    def c(v):
        if isinstance(v, dict):
            return {str(k).upper(): c(x) for k, x in v.items()}
        if isinstance(v, list):
            return [c(x) for x in v]
        if isinstance(v, (int, float)) and not isinstance(v, bool):
            return round(float(v), 1)
        if isinstance(v, str) and re.fullmatch(r"[A-Za-z]{3}\s*(->|→|–|-|/|to)\s*[A-Za-z]{3}", v.strip()):
            return norm_route(v)
        return v
    v = c(ans)
    if tid == "T6" and isinstance(v.get("TOP5"), list):
        v["TOP5"] = sorted(map(str, v["TOP5"]))
    return json.dumps(v, sort_keys=True)

# sanity check: reference answers score 1.0; list traps the grader can see (some traps only move
# un-graded fields, e.g. a count behind an unchanged route name)
visible = []
for t in TASKS:
    assert grade(t["id"], t["expected"], t["expected"])[0] == 1.0, t["id"]
    for name, ta in t["trap_answers"].items():
        if grade(t["id"], {**t["expected"], **ta}, t["expected"])[0] < 1.0:
            visible.append(f"{t['id']}:{name}")
print("grader self-check passed. traps that change the graded answer:", ", ".join(visible))

grader self-check passed. traps that change the graded answer: T1:no_dedup, T1:no_utc_fix, T1:no_midnight_wrap, T3:no_midnight_wrap, T3:naive_weather, T4:no_utc_fix, T4:no_midnight_wrap, T4:naive_weather, T5:no_midnight_wrap, T5:mean_of_carrier_rates, T6:union_of_carrier_top3, T8:no_midnight_wrap


## Experiment runner
`run_one` runs one system on one task with a fresh `RunLog`, grades it, and saves the full trace to
`traces/<system>_<task>_r<repeat>.json`. `run_suite` loops and appends every row to `results.csv`.

In [12]:
RESULTS_CSV = HW2 / "results.csv"

def trace_name(system, tid, repeat, effort="default"):
    return f"{system}_{tid}_r{repeat}" + ("" if effort == "default" else f"_{effort}") + ".json"

def run_one(system, task, llm, repeat=0, tag=""):
    log = RunLog(system, task["id"], repeat)
    t0 = time.time()
    try:
        final, metrics = SYSTEMS[system](task, llm, log)
    except Exception as e:                       # a crash is a failed run, not a crashed experiment
        final, metrics = None, {"error": repr(e)}
        log.event(system, "error", error=traceback.format_exc(limit=3))
    score, detail = grade(task["id"], final, task["expected"])
    row = dict(system=system + (f"+{tag}" if tag else ""), task=task["id"], category=task["category"],
               effort=REASONING_EFFORT or "default", repeat=repeat, score=round(score, 3), success=score >= 1.0, trap=match_trap(task, final, score),
               **log.totals(), wall_s=round(time.time() - t0, 1),
               **{k: (json.dumps(v) if isinstance(v, (list, dict)) else v) for k, v in metrics.items()})
    trace = dict(row=row, final=final, grade_detail=detail, calls=log.calls, messages=log.messages, events=log.events)
    (TRACE_DIR / trace_name(row["system"], task["id"], repeat, row["effort"])).write_text(json.dumps(trace, indent=1, default=str))
    return row

def run_suite(systems, task_ids, llm, repeats=1, tag="", start_repeat=0):
    rows = []
    for rep in range(start_repeat, start_repeat + repeats):
        for tid in task_ids:
            for system in systems:
                r = run_one(system, TASK_BY_ID[tid], llm, rep, tag)
                rows.append(r)
                print(f"{r['system']:>12} {tid} r{rep}  score={r['score']:.2f}  calls={r['calls']:>2}  "
                      f"tok={r['input_tokens']}/{r['output_tokens']}  ${r['cost_usd']:.4f}  {r['trap']}")
    df = pd.DataFrame(rows)
    # rows from different systems have different metric columns -> merge by name, never append blindly
    all_rows = pd.concat([pd.read_csv(RESULTS_CSV), df], ignore_index=True) if RESULTS_CSV.exists() else df
    all_rows.to_csv(RESULTS_CSV, index=False)
    return df

def show_trace(system, tid, repeat=0, effort="default", max_chars=600):
    # Compact view of a saved trajectory, for picking the 3 trajectories in part 6.
    tr = json.loads((TRACE_DIR / trace_name(system, tid, repeat, effort)).read_text())
    print(json.dumps(tr["row"], indent=1)[:1500])
    for ev in tr["events"]:
        body = {k: v for k, v in ev.items() if k not in ("agent", "kind")}
        print(f"\n--- [{ev['agent']}] {ev['kind']}\n{json.dumps(body, default=str)[:max_chars]}")
    print("\nFINAL:", tr["final"], "\nGRADE:", tr["grade_detail"])

## 1. Smoke test — one task, all three systems
Run this first to check the plumbing and measure cost per task before spending budget on the full suite.

In [12]:
# RUNS API
llm = LLMClient()
smoke = run_suite(list(SYSTEMS), ["T7"], llm, tag="smoke")
per_task_cost = smoke.groupby("system").cost_usd.mean()
print("\nprojected full-suite cost (8 tasks x 3 systems, 1 repeat): $%.2f" % (per_task_cost.sum() * 8))
smoke

single+smoke T7 r0  score=1.00  calls= 2  tok=1978/225  $0.0003  
hierarchy+smoke T7 r0  score=1.00  calls= 5  tok=4523/500  $0.0007  
debate+smoke T7 r0  score=1.00  calls=11  tok=11941/1375  $0.0019  

projected full-suite cost (8 tasks x 3 systems, 1 repeat): $0.02


,system,task,category,effort,repeat,score,success,trap,calls,input_tokens,...,caveats_passed,distinct_answers,solvers_no_result,solver_caps,solver_calls_used,solver_scores,critic_best,agg_chosen,final_is_majority,final_matches_a_solver
0,single+smoke,T7,wasteful,low,0,1.0,True,,2,1978,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,hierarchy+smoke,T7,wasteful,low,0,1.0,True,,5,4523,...,"[""cancelled"", ""dedup""]",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,debate+smoke,T7,wasteful,low,0,1.0,True,,11,11941,...,NaN,1.0,0.0,"[6, 9, 11]","[3, 4, 2]","[1.0, 1.0, 1.0]",S1,S1,True,True


## 2. Full run — all 8 tasks × 3 systems

In [13]:
# RUNS API
llm = LLMClient()
main = run_suite(["single", "hierarchy", "debate"], [t["id"] for t in TASKS], llm, repeats=1)

      single T1 r0  score=1.00  calls= 5  tok=12414/1720  $0.0021  
   hierarchy T1 r0  score=1.00  calls= 7  tok=15678/2881  $0.0030  


<string>:6: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
<string>:6: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
<string>:6: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
<string>:6: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
<string>:6: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
<string>:6: UserWarn

      debate T1 r0  score=1.00  calls=16  tok=48795/6218  $0.0080  
      single T2 r0  score=1.00  calls= 3  tok=7627/916  $0.0012  
   hierarchy T2 r0  score=1.00  calls=14  tok=23002/4529  $0.0046  
      debate T2 r0  score=1.00  calls=17  tok=57660/6614  $0.0091  
      single T3 r0  score=1.00  calls= 4  tok=10601/1098  $0.0016  
   hierarchy T3 r0  score=1.00  calls= 8  tok=24407/2209  $0.0035  
      debate T3 r0  score=1.00  calls=13  tok=33066/4061  $0.0053  
      single T4 r0  score=0.00  calls= 6  tok=17299/1599  $0.0025  
   hierarchy T4 r0  score=1.00  calls= 7  tok=16225/2227  $0.0027  
      debate T4 r0  score=1.00  calls=19  tok=63374/6218  $0.0094  
      single T5 r0  score=1.00  calls= 4  tok=11433/1118  $0.0017  
   hierarchy T5 r0  score=1.00  calls= 8  tok=22516/2594  $0.0035  
      debate T5 r0  score=1.00  calls=17  tok=49673/6039  $0.0080  
      single T6 r0  score=1.00  calls= 7  tok=20195/1518  $0.0028  
   hierarchy T6 r0  score=1.00  calls= 6  tok=1243

## 3. Results
Score and cost per system, per task, and per category; plus coordination measures.

In [14]:
res = pd.read_csv(RESULTS_CSV)
res = res[~res.system.str.contains(r"\+")]                 # main runs only (no smoke / surgery tags)
res["effort"] = res["effort"].fillna("default") if "effort" in res else "default"
res["cond"] = res.system + " @ " + res.effort                # e.g. "debate @ low"

print("Score by task (mean over repeats)")
display(res.pivot_table(index=["category", "task"], columns="cond", values="score", aggfunc="mean").round(2))

summary = res.groupby("cond").agg(
    mean_score=("score", "mean"), successes=("success", "sum"), runs=("score", "size"),
    calls=("calls", "mean"), in_tok=("input_tokens", "mean"), out_tok=("output_tokens", "mean"),
    cost_per_task=("cost_usd", "mean"), handoffs=("handoffs", "mean"), comm_chars=("comm_chars", "mean"),
).round(3)
print("\nPerformance and cost per system (per-task means)")
display(summary)

print("\nScore by category")
display(res.pivot_table(index="category", columns="cond", values="score", aggfunc="mean").round(2))

print("\nWhich trap explained each failure")
display(res[res.score < 1][["cond", "task", "score", "trap"]].sort_values(["task", "cond"]))

Score by task (mean over repeats)


cond                 debate @ default  debate @ low  hierarchy @ default  \
category       task                                                        
distinct_roles T3                1.00           1.0                  1.0   
               T4                1.00           1.0                  1.0   
integration    T5                1.00           1.0                  1.0   
               T6                1.00           1.0                  1.0   
separable      T1                1.00           1.0                  1.0   
               T2                0.92           1.0                  1.0   
shared_mistake T8                1.00           0.0                  1.0   
wasteful       T7                1.00           1.0                  1.0   

cond                 hierarchy @ low  single @ default  single @ low  
category       task                                                   
distinct_roles T3                1.0               1.0           1.0  
               T4                1.0               1.0           0.0  
integration    T5                1.0               1.0           1.0  
               T6                1.0               1.0           1.0  
separable      T1                1.0               1.0           1.0  
               T2                1.0               1.0           1.0  
shared_mistake T8                1.0               1.0           0.0  
wasteful       T7                1.0               1.0           1.0


Performance and cost per system (per-task means)


,mean_score,successes,runs,calls,in_tok,out_tok,cost_per_task,handoffs,comm_chars
cond,,,,,,,,,
debate @ default,0.990,7,8,17.375,57601.000,7278.875,0.009,3.000,12929.750
debate @ low,0.875,7,8,15.625,45501.875,5137.125,0.007,3.000,12233.875
hierarchy @ default,1.000,8,8,10.875,40667.250,4025.000,0.006,2.375,4754.875
hierarchy @ low,1.000,8,8,8.125,20085.375,2598.125,0.003,2.250,4465.250
single @ default,1.000,8,8,5.125,18200.750,1785.875,0.003,0.000,0.000
single @ low,0.750,6,8,4.500,11774.375,1176.250,0.002,0.000,0.000



Score by category


cond,debate @ default,debate @ low,hierarchy @ default,hierarchy @ low,single @ default,single @ low
category,,,,,,
distinct_roles,1.00,1.0,1.0,1.0,1.0,0.5
integration,1.00,1.0,1.0,1.0,1.0,1.0
separable,0.96,1.0,1.0,1.0,1.0,1.0
shared_mistake,1.00,0.0,1.0,1.0,1.0,0.0
wasteful,1.00,1.0,1.0,1.0,1.0,1.0



Which trap explained each failure


,cond,task,score,trap
8,debate @ default,T2,0.917,NaN
39,single @ low,T4,0.000,NaN
53,debate @ low,T8,0.000,NaN
51,single @ low,T8,0.000,no_midnight_wrap


In [15]:
coord_cols = {"hierarchy": ["task", "n_workers", "worker_caps", "worker_calls_used", "files_overlap", "workers_no_result",
                            "caveats_passed", "invalid_files"],
              "debate": ["task", "solver_caps", "solver_calls_used", "distinct_answers", "solver_scores", "critic_best",
                         "agg_chosen", "final_is_majority"],
              "single": ["task", "calls", "code_runs"]}
for s, cols in coord_cols.items():
    print(f"\nCoordination / process measures: {s}")
    display(res[res.system == s][["effort"] + [c for c in cols if c in res.columns] + ["score"]])


Coordination / process measures: hierarchy


,effort,task,n_workers,worker_caps,worker_calls_used,files_overlap,workers_no_result,caveats_passed,invalid_files,score
4,default,T1,1.0,[18],[7],0.0,0.0,"[""threshold_15"", ""cancelled"", ""utc"", ""dedup"", ...",[],1.0
7,default,T2,2.0,"[9, 14]","[4, 10]",2.0,0.0,"[""threshold_15"", ""cancelled"", ""utc"", ""dedup"", ...",[],1.0
10,default,T3,2.0,"[9, 10]","[8, 7]",7.0,0.0,"[""threshold_15"", ""cancelled"", ""dedup"", ""midnig...",[],1.0
13,default,T4,1.0,[18],[8],0.0,0.0,"[""threshold_15"", ""cancelled"", ""utc"", ""dedup"", ...",[],1.0
16,default,T5,1.0,[18],[7],0.0,0.0,"[""threshold_15"", ""cancelled"", ""utc"", ""dedup"", ...",[],1.0
19,default,T6,2.0,"[9, 15]","[3, 6]",8.0,0.0,"[""cancelled"", ""utc"", ""dedup""]",[],1.0
22,default,T7,1.0,[18],[3],0.0,0.0,"[""cancelled"", ""dedup""]",[],1.0
25,default,T8,1.0,[18],[8],0.0,0.0,"[""cancelled"", ""utc"", ""dedup"", ""midnight""]",[],1.0
31,low,T1,1.0,[18],[5],0.0,0.0,"[""threshold_15"", ""cancelled"", ""utc"", ""dedup"", ...",[],1.0
34,low,T2,3.0,"[6, 6, 9]","[6, 3, 3]",4.0,0.0,"[""threshold_15"", ""cancelled"", ""utc"", ""midnight""]",[],1.0



Coordination / process measures: debate


,effort,task,solver_caps,solver_calls_used,distinct_answers,solver_scores,critic_best,agg_chosen,final_is_majority,score
5,default,T1,"[6, 6, 7]","[6, 5, 6]",1.0,"[1.0, 1.0, 1.0]",S3,S3,True,1.000
8,default,T2,"[6, 6, 6]","[6, 6, 6]",2.0,"[0.9166666666666666, 0.9166666666666666, 1.0]",S1,S1,True,0.917
11,default,T3,"[6, 7, 9]","[5, 4, 5]",1.0,"[1.0, 1.0, 1.0]",S2,S2,True,1.000
14,default,T4,"[6, 7, 7]","[5, 6, 7]",2.0,"[0.5, 1.0, 1.0]",S2,S2,True,1.000
17,default,T5,"[6, 6, 6]","[6, 6, 6]",2.0,"[1.0, 1.0, 1.0]",S3,S3,True,1.000
20,default,T6,"[6, 7, 9]","[5, 4, 6]",1.0,"[1.0, 1.0, 1.0]",S3,S3,True,1.000
23,default,T7,"[6, 10, 14]","[2, 2, 4]",1.0,"[1.0, 1.0, 1.0]",S2,S2,True,1.000
26,default,T8,"[6, 6, 8]","[6, 4, 5]",1.0,"[1.0, 1.0, 1.0]",S2,S2,True,1.000
32,low,T1,"[6, 8, 9]","[4, 5, 5]",3.0,"[1.0, 0.5, 0.8333333333333334]",S1,S1,False,1.000
35,low,T2,"[6, 7, 8]","[5, 5, 5]",1.0,"[1.0, 1.0, 1.0]",S1,S1,True,1.000



Coordination / process measures: single


,effort,task,calls,code_runs,score
3,default,T1,6,5.0,1.0
6,default,T2,7,6.0,1.0
9,default,T3,5,4.0,1.0
12,default,T4,6,5.0,1.0
15,default,T5,4,3.0,1.0
18,default,T6,5,4.0,1.0
21,default,T7,2,1.0,1.0
24,default,T8,6,5.0,1.0
30,low,T1,5,4.0,1.0
33,low,T2,3,2.0,1.0


In [20]:
try:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(5, 3.5))
    for s, g in summary.iterrows():
        x = g.cost_per_task if pd.notna(g.cost_per_task) else g.in_tok + g.out_tok
        ax.scatter(x, g.mean_score, s=80)
        ax.annotate(s, (x, g.mean_score), textcoords="offset points", xytext=(6, 4))
    ax.set_xlabel("mean cost per task ($)" if summary.cost_per_task.notna().all() else "mean tokens per task")
    ax.set_ylabel("mean score")
    ax.set_title("Performance vs cost")
    fig.tight_layout()
    fig.savefig(HW2 / "perf_vs_cost.png", dpi=200)
    plt.show()
except ImportError:
    print("pip install matplotlib for the plot (the summary table above is enough for the report)")

pip install matplotlib for the plot (the summary table above is enough for the report)


## 4. Stability — repeat three close or surprising comparisons
Fill in after looking at the main results.

In [21]:
# RUNS API
REPEAT = [("single", "T4"), ("hierarchy", "T4"), ("debate", "T4"),
          ("single", "T8"), ("hierarchy", "T8"), ("debate", "T8")]  # <- edit after the main run
llm = LLMClient()
for system, tid in REPEAT:
    run_suite([system], [tid], llm, repeats=2, start_repeat=1)       # repeats 1-2 (+ main run = 3 samples)

      single T4 r1  score=1.00  calls= 5  tok=13562/1417  $0.0021  
      single T4 r2  score=1.00  calls= 6  tok=19425/1727  $0.0028  
   hierarchy T4 r1  score=1.00  calls= 6  tok=13220/2463  $0.0026  
   hierarchy T4 r2  score=1.00  calls= 8  tok=18683/2357  $0.0030  


<string>:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
<string>:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
<string>:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
<string>:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.


      debate T4 r1  score=1.00  calls=19  tok=59708/5446  $0.0087  
      debate T4 r2  score=0.00  calls=20  tok=56395/5772  $0.0085  
      single T8 r1  score=1.00  calls= 7  tok=25509/1902  $0.0035  
      single T8 r2  score=1.00  calls= 4  tok=9224/1085  $0.0015  
   hierarchy T8 r1  score=0.00  calls= 8  tok=30038/2585  $0.0043  
   hierarchy T8 r2  score=1.00  calls= 8  tok=27798/3133  $0.0043  


<string>:5: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
<string>:5: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
<string>:5: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
<string>:5: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
<string>:5: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
<string>:5: UserWarn

      debate T8 r1  score=1.00  calls=16  tok=39615/4895  $0.0064  
      debate T8 r2  score=1.00  calls=20  tok=67935/5292  $0.0094  


## 5. Topology surgery
Two structural changes, each run at low effort, 3 repeats, tagged `+surgery` (kept out of the main tables).

**#1 `debate_vote` — remove the critic and aggregator.** Same 3 independent solvers, but the final answer is a
plain plurality vote over their answers (ties → S1). Tests whether the LLM review layer helps or hurts: in the
main runs the critic/aggregator overrode a correct majority (T8 r0) and picked an over-wrapped answer (T4 r2), but
also rescued T1 when all three solvers disagreed. Tasks: T1, T2, T4, T8.

**#4 `hierarchy_split` — force data-parallel decomposition.** The manager may no longer use a single worker; it must
use exactly 3 workers and partition the six airline files between them. Everything else (prompts, pool, rollover,
manager synthesis) is unchanged. Tests the integration failures the T5/T6 tasks were built for (averaging carrier
rates, union of per-carrier top lists), which never fired because the manager kept everything in one worker.
Tasks: T2, T5, T6.

In [13]:
# RUNS API
assert REASONING_EFFORT == "low"     # compare against the low-effort baselines
SURGERY = {"debate_vote": ["T1", "T2", "T4", "T8"], "hierarchy_split": ["T2", "T5", "T6"]}
llm = LLMClient()
for system, tids in SURGERY.items():
    run_suite([system], tids, llm, repeats=3, tag="surgery")

debate_vote+surgery T1 r0  score=0.00  calls=14  tok=39873/3892  $0.0059  
debate_vote+surgery T2 r0  score=1.00  calls=12  tok=31103/3296  $0.0048  
debate_vote+surgery T4 r0  score=1.00  calls=17  tok=48867/4346  $0.0071  
debate_vote+surgery T8 r0  score=1.00  calls=15  tok=49559/4386  $0.0071  
debate_vote+surgery T1 r1  score=0.50  calls=16  tok=49721/5401  $0.0077  no_midnight_wrap
debate_vote+surgery T2 r1  score=1.00  calls=14  tok=44014/4208  $0.0065  
debate_vote+surgery T4 r1  score=1.00  calls=18  tok=51189/5182  $0.0077  
debate_vote+surgery T8 r1  score=1.00  calls=15  tok=44888/4425  $0.0067  
debate_vote+surgery T1 r2  score=0.83  calls=18  tok=50214/5078  $0.0076  no_utc_fix
debate_vote+surgery T2 r2  score=1.00  calls=17  tok=63863/5542  $0.0092  
debate_vote+surgery T4 r2  score=1.00  calls=17  tok=45351/4802  $0.0069  
debate_vote+surgery T8 r2  score=1.00  calls=17  tok=57323/5402  $0.0084  


<string>:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
<string>:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.


hierarchy_split+surgery T2 r0  score=1.00  calls=18  tok=46089/6797  $0.0080  
hierarchy_split+surgery T5 r0  score=0.00  calls=16  tok=32195/5407  $0.0059  
hierarchy_split+surgery T6 r0  score=1.00  calls=13  tok=19834/4373  $0.0042  
hierarchy_split+surgery T2 r1  score=0.58  calls=16  tok=35482/9938  $0.0085  
hierarchy_split+surgery T5 r1  score=0.00  calls=15  tok=29249/4517  $0.0052  
hierarchy_split+surgery T6 r1  score=1.00  calls=12  tok=13655/3137  $0.0029  
hierarchy_split+surgery T2 r2  score=1.00  calls=19  tok=47873/6019  $0.0078  
hierarchy_split+surgery T5 r2  score=1.00  calls=20  tok=53970/6297  $0.0085  
hierarchy_split+surgery T6 r2  score=0.80  calls=11  tok=11043/2569  $0.0024  


In [14]:
# Surgery vs its baseline (same tasks, low effort). Baselines have 3 repeats on T4/T8, 1 elsewhere.
allr = pd.read_csv(RESULTS_CSV)
allr["effort"] = allr["effort"].fillna("default")
low = allr[allr.effort == "low"].copy()
low["system"] = low.system.str.replace("+surgery", "", regex=False)
PAIRS = {"debate_vote": "debate", "hierarchy_split": "hierarchy"}
for surg, base in PAIRS.items():
    tids = SURGERY[surg]
    sub = low[low.system.isin([surg, base]) & low.task.isin(tids)]
    print(f"\n=== {base}  vs  {surg}")
    display(sub.groupby(["task", "system"]).agg(
        runs=("score", "size"), mean_score=("score", "mean"), calls=("calls", "mean"),
        cost=("cost_usd", "mean"), traps=("trap", lambda s: ", ".join(sorted(set(s.dropna().astype(str)))))
    ).unstack("system").round(4))
    print(sub.groupby("system").agg(mean_score=("score", "mean"), cost=("cost_usd", "mean")).round(4))

cols = {"debate_vote": ["task", "repeat", "solver_scores", "distinct_answers", "vote_count", "vote_tie", "score", "trap"],
        "hierarchy_split": ["task", "repeat", "split_ok", "n_workers", "airlines_per_worker", "worker_calls_used",
                            "caveats_passed", "score", "trap"]}
for s, c in cols.items():
    print(f"\nProcess measures: {s}")
    d = allr[allr.system == s + "+surgery"]
    display(d[[x for x in c if x in d.columns]])


=== debate  vs  debate_vote


runs             mean_score                calls                cost  \
system debate debate_vote     debate debate_vote   debate debate_vote  debate   
task                                                                            
T1          1           3     1.0000      0.4443  16.0000     16.0000  0.0080   
T2          1           3     1.0000      1.0000  17.0000     14.3333  0.0091   
T4          3           3     0.6667      1.0000  19.3333     17.3333  0.0089   
T8          3           3     0.6667      1.0000  18.3333     15.6667  0.0084   

                    traps                                
system debate_vote debate                   debate_vote  
task                                                     
T1          0.0071         no_midnight_wrap, no_utc_fix  
T2          0.0068                                       
T4          0.0072                                       
T8          0.0074

             mean_score    cost
system                         
debate           0.7500  0.0086
debate_vote      0.8611  0.0071

=== hierarchy  vs  hierarchy_split


runs                 mean_score                     calls  \
system hierarchy hierarchy_split  hierarchy hierarchy_split hierarchy   
task                                                                    
T2             1               3        1.0          0.8610      14.0   
T5             1               3        1.0          0.3333       8.0   
T6             1               3        1.0          0.9333       6.0   

                            cost                     traps                  
system hierarchy_split hierarchy hierarchy_split hierarchy hierarchy_split  
task                                                                        
T2             17.6667    0.0046          0.0081                            
T5             17.0000    0.0035          0.0066                            
T6             12.0000    0.0022          0.0032

                 mean_score    cost
system                             
hierarchy            1.0000  0.0034
hierarchy_split      0.7092  0.0059

Process measures: debate_vote


,task,repeat,solver_scores,distinct_answers,vote_count,vote_tie,score,trap
66,T1,0,"[1.0, 0.0, 0.0]",2.0,2.0,False,0.000,NaN
67,T2,0,"[1.0, 1.0, 1.0]",1.0,3.0,False,1.000,NaN
68,T4,0,"[1.0, 1.0, 1.0]",1.0,3.0,False,1.000,NaN
69,T8,0,"[1.0, 1.0, 1.0]",1.0,3.0,False,1.000,NaN
70,T1,1,"[1.0, 0.5, 0.5]",2.0,2.0,False,0.500,no_midnight_wrap
71,T2,1,"[0.9166666666666666, 1.0, 1.0]",2.0,2.0,False,1.000,NaN
72,T4,1,"[1.0, 1.0, 1.0]",1.0,3.0,False,1.000,NaN
73,T8,1,"[1.0, 1.0, 1.0]",2.0,2.0,False,1.000,NaN
74,T1,2,"[0.8333333333333334, 0.0, 0.0]",3.0,1.0,True,0.833,no_utc_fix
75,T2,2,"[1.0, 1.0, 1.0]",1.0,3.0,False,1.000,NaN



Process measures: hierarchy_split


,task,repeat,split_ok,n_workers,airlines_per_worker,worker_calls_used,caveats_passed,score,trap
78,T2,0,True,3.0,"[2, 2, 2]","[4, 6, 6]","[""threshold_15"", ""cancelled"", ""utc"", ""midnight""]",1.000,NaN
79,T5,0,True,3.0,"[2, 2, 2]","[4, 7, 3]","[""threshold_15"", ""cancelled"", ""utc"", ""dedup"", ...",0.000,NaN
80,T6,0,True,3.0,"[2, 2, 2]","[4, 3, 4]","[""cancelled"", ""utc"", ""dedup""]",1.000,NaN
81,T2,1,True,3.0,"[2, 2, 2]","[4, 4, 6]","[""threshold_15"", ""cancelled"", ""utc"", ""midnight""]",0.583,NaN
82,T5,1,True,3.0,"[2, 2, 2]","[3, 3, 7]","[""threshold_15"", ""cancelled"", ""utc"", ""midnight""]",0.000,NaN
83,T6,1,True,3.0,"[2, 2, 2]","[4, 3, 3]","[""cancelled"", ""utc""]",1.000,NaN
84,T2,2,True,3.0,"[2, 2, 2]","[4, 7, 6]","[""threshold_15"", ""cancelled"", ""utc"", ""dedup"", ...",1.000,NaN
85,T5,2,True,3.0,"[2, 2, 2]","[6, 6, 6]","[""threshold_15"", ""cancelled"", ""utc"", ""dedup"", ...",1.000,NaN
86,T6,2,True,3.0,"[2, 2, 2]","[3, 3, 3]","[""cancelled"", ""utc"", ""dedup""]",0.800,NaN


## 6. Trajectories for the report

In [ ]:
# show_trace("hierarchy", "T5", 0)